# NWS Forecast Zones Feeder (Fabric Notebook)

Polls NWS land-zone forecasts (`api.weather.gov/zones/forecast/{zoneId}/forecast`) and marine-zone bulletins (`tgftp.nws.noaa.gov/data/forecasts/marine/coastal/pz/{zone}.txt`) for a configured set of forecast zones and emits CloudEvents to the bound Fabric Event Stream. Zone reference metadata is published first and refreshed periodically; only changed forecast snapshots are re-emitted (dedupe by `updated` timestamp for land zones, by bulletin content hash for marine zones).

**Runtime model**
- The `nws_forecasts` package (with both generated producer sub-packages) is pre-installed by the **attached Fabric Environment**. No pip install is required at runtime.
- The Event Stream connection string is **looked up at runtime** via the Fabric REST API using the notebook's workspace identity (or user identity). The deploy script does not bake any secret into the notebook.
- The bridge runs `nws-forecasts --once`, exits after one polling cycle, and persists dedupe state to a Lakehouse Files path so the next scheduled run only emits changed forecasts.

In [ ]:
# === PARAMETERS (overwritten by deploy-feeder-notebook.ps1) ===
EVENTSTREAM_NAME = "nws-forecasts-ingest"     # Fabric Event Stream item name in this workspace
STATE_FILE       = "/lakehouse/default/Files/feeder-state/nws-forecasts/state.json"
POLLING_INTERVAL = 900                       # seconds; only used if ONCE_MODE is False
ONCE_MODE        = True                      # True for scheduled execution
WORKSPACE_ID     = ""                        # filled in by deploy script (optional; falls back to runtime context)

## Look up the Event Stream connection string
Uses `notebookutils.credentials.getToken('pbi')` (workspace identity when scheduled, user identity when run interactively) to call the Fabric REST + ES MWC APIs and retrieve the custom-endpoint `primaryConnectionString`. The token is short-lived and never persisted.

In [ ]:
import os, time, json
import requests

FABRIC_API = 'https://api.fabric.microsoft.com/v1'

def _get_pbi_token():
    try:
        import notebookutils  # noqa: F401
        return notebookutils.credentials.getToken('pbi')
    except Exception:
        from notebookutils import mssparkutils
        return mssparkutils.credentials.getToken('pbi')

def _resolve_workspace_id(explicit: str) -> str:
    if explicit:
        return explicit
    try:
        import notebookutils
        ctx = notebookutils.runtime.context
        return ctx['currentWorkspaceId']
    except Exception:
        pass
    from notebookutils import mssparkutils
    ctx = json.loads(mssparkutils.runtime.context)
    return ctx['currentWorkspaceId']

def lookup_es_connection_string(workspace_id: str, eventstream_name: str) -> str:
    """Return the primary connection string for the named Event Stream's custom-endpoint source.

    Uses the public Fabric Topology API (2 calls). Documented at
    https://learn.microsoft.com/en-us/rest/api/fabric/eventstream/topology/get-eventstream-source-connection
    """
    headers = {'Authorization': f'Bearer {_get_pbi_token()}'}

    es_list = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams', headers=headers, timeout=30)
    es_list.raise_for_status()
    es = next((x for x in es_list.json().get('value', []) if x['displayName'] == eventstream_name), None)
    if not es:
        raise RuntimeError(f"Event Stream '{eventstream_name}' not found in workspace {workspace_id}.")

    topo = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams/{es["id"]}/topology', headers=headers, timeout=30)
    topo.raise_for_status()
    src = next((s for s in topo.json().get('sources', []) if s.get('type') == 'CustomEndpoint'), None)
    if not src:
        raise RuntimeError("Event Stream has no CustomEndpoint source.")

    last_err = None
    for attempt in range(5):
        try:
            conn = requests.get(
                f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams/{es["id"]}/sources/{src["id"]}/connection',
                headers=headers, timeout=30,
            )
            conn.raise_for_status()
            cs = conn.json().get('accessKeys', {}).get('primaryConnectionString')
            if cs:
                return cs
            last_err = 'empty primaryConnectionString'
        except Exception as e:
            last_err = str(e)
        time.sleep(min(15, 3 * (attempt + 1)))
    raise RuntimeError(f'Could not retrieve connection string after 5 attempts: {last_err}')

ws_id = _resolve_workspace_id(WORKSPACE_ID)
cs = lookup_es_connection_string(ws_id, EVENTSTREAM_NAME)
print(f"Workspace:        {ws_id}")
print(f"Event Stream:     {EVENTSTREAM_NAME}")
print(f"Connection ready: length={len(cs)}")
os.environ['CONNECTION_STRING'] = cs

## Prepare state and run one polling cycle

In [ ]:
import os, pathlib, sys, traceback, datetime

LOG_PATH = '/lakehouse/default/Files/feeder-state/nws-forecasts/last-run.log'
pathlib.Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)

def _log(msg):
    line = f'[{datetime.datetime.utcnow().isoformat()}Z] {msg}'
    print(line)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

with open(LOG_PATH, 'w', encoding='utf-8') as f:
    f.write('')

try:
    _log('Starting feeder cycle')
    pathlib.Path(STATE_FILE).parent.mkdir(parents=True, exist_ok=True)
    os.environ['STATE_FILE']       = STATE_FILE
    os.environ['NWS_FORECAST_STATE_FILE'] = STATE_FILE
    os.environ['POLLING_INTERVAL'] = str(POLLING_INTERVAL)
    os.environ['NWS_FORECAST_POLL_INTERVAL_SECONDS'] = str(POLLING_INTERVAL)
    os.environ['ONCE_MODE']        = 'true' if ONCE_MODE else 'false'
    _log(f'CONNECTION_STRING present: {bool(os.environ.get("CONNECTION_STRING"))}')

    _log('Importing nws_forecasts package from Fabric Environment...')
    from nws_forecasts import nws_forecasts as feeder
    _log(f'Imported feeder from: {feeder.__file__}')

    argv_backup = sys.argv
    try:
        sys.argv = ['nws-forecasts', '--once'] if ONCE_MODE else ['nws-forecasts']
        _log(f'Running feeder.main() with argv={sys.argv}')
        import threading
        _err = []
        def _worker():
            try:
                feeder.main()
            except BaseException as e:
                _err.append(e)
        t = threading.Thread(target=_worker, daemon=True)
        t.start()
        t.join()
        if _err:
            raise _err[0]
    finally:
        sys.argv = argv_backup
    _log('Cycle complete.')
    try:
        import notebookutils
        notebookutils.notebook.exit('OK')
    except Exception:
        pass
except Exception as exc:
    tb = traceback.format_exc()
    _log(f'FAILED: {exc}\n{tb}')
    try:
        import notebookutils
        notebookutils.notebook.exit(f'FAIL: {exc}')
    except Exception:
        pass
    raise